In [12]:
#1 Instalasi dan Import Library

!pip install --quiet gensim numpy pandas scikit-learn
import re
import numpy as np
import pandas as pd
from collections import Counter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 49.1 MB/s eta 0:00:00


In [13]:
#2 Menyiapkan corpus

paragraf_1 =("When my family and I were driving through France a few years ago, we were enchanted by the hundreds of storybook cows grazing on picturesque pastures right next to the highway. Then, within twenty minutes, we started ignoring the cows. The new cows were just like the old cows, and what once was amazing was now common. Worse than common. It was boring. Cows, after you’ve seen them for a while, are boring. A Purple Cow, though. Now that would be interesting. The essence of the Purple Cow is that it must be remarkable")
paragraf_2 =("Something remarkable is worth talking about. Worth noticing. Exceptional. New. Interesting. It’s a Purple Cow. Boring stuff is invisible. It’s a brown cow. Remarkable marketing is the art of building things worth noticing right into your product or service.")

corpus_raw = paragraf_1 + " " + paragraf_2
print(corpus_raw)

When my family and I were driving through France a few years ago, we were enchanted by the hundreds of storybook cows grazing on picturesque pastures right next to the highway. Then, within twenty minutes, we started ignoring the cows. The new cows were just like the old cows, and what once was amazing was now common. Worse than common. It was boring. Cows, after you’ve seen them for a while, are boring. A Purple Cow, though. Now that would be interesting. The essence of the Purple Cow is that it must be remarkable Something remarkable is worth talking about. Worth noticing. Exceptional. New. Interesting. It’s a Purple Cow. Boring stuff is invisible. It’s a brown cow. Remarkable marketing is the art of building things worth noticing right into your product or service.


In [14]:
import re

#SOAL 1

#Berapa jumlah total token setelah tokenisasi? Case Folding - mengubah semua huruf menjadi huruf kecil.
teks_folded = corpus_raw.lower ()
teks_no_apostrof = teks_folded.replace("'", "")
teks_bersih = re.sub(r"[^a-z\s]", " ", teks_no_apostrof)

#Tokenisasi - memecah teks bersih menjadi daftar token berdasarkan spasi.
def preprocess(text):
  text = text.lower()
  text = text.replace("'", "")
  text = re.sub(r"[^a-z\s]", " ", text)
  tokens = text.split()
  return tokens

tokens = preprocess(corpus_raw)
print("Jumlah total token : ", len(tokens))
print("Jumlah token unik (ukuran vocabulary):", len(set(tokens)))

# Tampilkan 10 kata dengan frekuensi kemunculan tertinggi (word frequency).
freq=Counter(tokens)
for w, c in freq.most_common(10):
  print(f" {w:<12} {c}")

  #Apakah terdapat stopword bahasa Inggris di antara 10 kata dengan frekuensi tertinggi? Menurut Anda, apakah stopword perlu dihapus sebelum melatih Word2Vec pada Soal 3? Jelaskan berdasarkan tujuan word embedding.

Jumlah total token :  136
Jumlah token unik (ukuran vocabulary): 87
 the          8
 a            5
 cows         5
 it           4
 cow          4
 is           4
 were         3
 of           3
 was          3
 boring       3


In [15]:
#S O AL 2 - C B OW d a n S k i p -G r a m M a n u a l
#a. Tentukan seluruh training pairs untuk metode CBOW (Context → Target) dari kalimat di atas.
def cbow_pairs(tokens, window):
  pairs =[]
  for i, target in enumerate(tokens):
    context = [tokens [j] for j in range (max (0, i-window), min(len (tokens), i+window+1)) if j != i]
    pairs.append((context, target))
  return pairs

def skipgram_pairs(tokens, window):
  pairs = []
  for i, target in enumerate(tokens):
    for j in range(max(0, i - window), min(len(tokens), i + window + 1)):
       if j != i:
        pairs.append((target, tokens[j]))
    return pairs

kalimat = ["Boring", "Boring", "is", "invisible"]
print(cbow_pairs(kalimat, window=1))
print(skipgram_pairs(kalimat, window=1))

#b. Tentukan seluruh training pairs untuk metode Skip-gram (Target → Context) dari kalimat di atas.

#c. Sajikan jawabanmu dalam bentuk tabel, mengikuti format yang dicontohkan pada materi kuliah pertemuan ini.

[(['Boring'], 'Boring'), (['Boring', 'is'], 'Boring'), (['Boring', 'invisible'], 'is'), (['is'], 'invisible')]
[('Boring', 'Boring')]


In [16]:
!pip install --quiet gensim
#S O AL 3 – M e l a t i h W o r d 2 v e c d e n g a n G e n s im
#Menggunakan seluruh corpus (Paragraf 1 + Paragraf 2) yang sudah dipreprocessing pada Soal 1:
#Menggunakan seluruh corpus (Paragraf 1 + Paragraf 2) yang sudah dipreprocessing pada Soal 1:
from gensim.models import Word2Vec
sentences = [tokens] #tokens dari langkah diatasnya

model_cbow = Word2Vec(sentences, vector_size=20, window=2, min_count=1, sg=0, epochs=200, seed=42)
model_sg = Word2Vec(sentences, vector_size=20, window=2, min_count=1, sg=1, epochs=200, seed=42)

for kata in ["boring", "remarkable", "cow"]:
  print(kata, "Cbow: ", model_cbow.wv.most_similar(kata, topn=3))
  print(kata, "Skipgram: ", model_sg.wv.most_similar(kata, topn=3))


#❖ Tampilkan 3 kata paling mirip (most_similar) untuk kata “boring”, “remarkable”, dan “cow” pada masing-masing model.

#❖ Bandingkan hasil CBOW vs Skip-gram: mana yang menurutmu lebih masuk akal untuk corpus sekecil ini? Kaitkan jawabanmu dengan sifat masing-masing arsitektur.

boring Cbow:  [('is', 0.9971520900726318), ('the', 0.9961457252502441), ('cow', 0.9956957101821899)]
boring Skipgram:  [('purple', 0.9912140369415283), ('it', 0.9903478026390076), ('for', 0.9902265071868896)]
remarkable Cbow:  [('boring', 0.9936053156852722), ('cow', 0.9935083985328674), ('art', 0.9929289221763611)]
remarkable Skipgram:  [('cow', 0.9886171817779541), ('is', 0.9885520339012146), ('brown', 0.9881460666656494)]
cow Cbow:  [('boring', 0.9956958293914795), ('then', 0.9956409335136414), ('we', 0.9947924017906189)]
cow Skipgram:  [('stuff', 0.9930018186569214), ('s', 0.9901117086410522), ('is', 0.9896367788314819)]


In [17]:
#S O AL 4 – M a t r i k s C o - o c c u r r e n c e s e b a g a i D a s a r G l o Ve
#❖ Bangun matriks co-occurrence dari mini-corpus berikut dengan window size = 1.
sentences_mini =[
    ["Cows", "are", "boring"],
    ["A", "Purple", "Cow", "is", "remarkable"],
    ["Boring", "stuff", "is", "invisible"],
]

vocab=sorted(set(w for s in sentences_mini for w in s))
idx= {w: i for i, w in enumerate(vocab)}
mat = np.zeros((len(vocab), len(vocab)), dtype=int)

window = 1
for s in sentences_mini:
  for i, w in enumerate(s):
    for j in range(max(0, i-window), min(len(s), i+window+1)):
      if i != j:
        mat[idx[w], idx[s[j]]] += 1

co_df = pd.DataFrame(mat, columns=vocab, index=vocab)
print(co_df)

#❖ Kata apa yang paling sering “berdekatan” dengan kata “boring”?
#❖ Apakah pola co-occurrence ini konsisten dengan makna kata “boring” dalam corpus (bandingkan dengan hasil most_similar pada Soal 3)?



            A  Boring  Cow  Cows  Purple  are  boring  invisible  is  \
A           0       0    0     0       1    0       0          0   0   
Boring      0       0    0     0       0    0       0          0   0   
Cow         0       0    0     0       1    0       0          0   1   
Cows        0       0    0     0       0    1       0          0   0   
Purple      1       0    1     0       0    0       0          0   0   
are         0       0    0     1       0    0       1          0   0   
boring      0       0    0     0       0    1       0          0   0   
invisible   0       0    0     0       0    0       0          0   1   
is          0       0    1     0       0    0       0          1   0   
remarkable  0       0    0     0       0    0       0          0   1   
stuff       0       1    0     0       0    0       0          0   1   

            remarkable  stuff  
A                    0      0  
Boring               0      1  
Cow                  0      0  
Cows   

In [18]:
import numpy as np
#S O AL 5 – M e n g h i t u n g C o s i n e S i m i l a r i t y
#Diketahui embedding 2 dimensi ilustratif untuk kata “remarkable” dan “invisible”.
#❖ Hitung cosine similarity kedua vektor.
A = np.array([0.9, 0.6])
B = np.array([-0.7,-0.5])

cosine = np.dot(A,B) / (np.linalg.norm(A) * np.linalg.norm(B))
print(f"cos(theta) = {cosine:.4f}")

#❖ Interpretasikan nilai yang diperoleh berdasarkan arah kedua vektor.
#❖ Apakah cosine similarity rendah atau negatif selalu berarti dua kata merupakan antonim? Jelaskan
#❖ Dalam konteks Purple Cow, bagaimana hubungan konseptual antara “remarkable” dan “invisible”?

cos(theta) = -0.9995


In [22]:
#S O AL 6 – F a s t-T e x t & O O V
#Pilih atau buat 1 kata yang sengaja TIDAK muncul pada corpus (misalnya “remarkably”, “cowish”, atau typo seperti “borring”).
#a. Coba cari vektor kata tersebut menggunakan model Word2Vec dari Soal 3. Apa yang terjadi?
#b. Latih model FastText (gensim) pada corpus yang sama, lalu coba lagi kata tersebut. Apakah FastText tetap bisa menghasilkan vektor?
kata_oov = "remarkably"
## Word2Vec akan gagal (KeyError) karena kata ini tidak ada di
#vocabulary
try:
  print(model_cbow.wv[kata_oov])
except KeyError as e:
  print("KeyError: ", e)

# FastText tetap bisa membentuk vektor lewat character n-gram
from gensim.models import FastText
model_ft = FastText(sentences, vector_size = 20, window = 2, min_count = 1, sg=1, epochs = 200, seed = 42, min_n=2, max_n=4)

print(model_ft.wv[kata_oov][:5])
print(model_ft.wv.most_similar(kata_oov, topn=3))
#c. Jelaskan mengapa hal ini bisa terjadi, dikaitkan dengan cara kerja FastText memecah kata menjadi sub-kata (character n-gram).

KeyError:  "Key 'remarkably' not present"
[-0.1369718   0.35269484 -0.0872735   0.33531457 -0.197575  ]
[('remarkable', 0.9998920559883118), ('invisible', 0.99964839220047), ('interesting', 0.9995867013931274)]


In [23]:
#S O AL 7 – A n a l i s i s
#Seth Godin berargumen bahwa produk yang “boring” itu invisible (tidak diperhatikan), sedangkan yang “remarkable” menyebar dengan sendirinya.
#Berdasarkan hasil eksperimen pada Soal 3–5, tuliskan analisis yang menjawab:
#• Apakah representasi vektor yang dihasilkan mencerminkan hubungan semantik antara “boring”, “remarkable”, dan kata-kata terkait dalam corpus? Jelaskan berdasarkan hasil Word2Vec, co-occurrence, dan cosine similarity.
#• Apa keterbatasan Word2Vec/GloVe dalam menangkap hubungan semantik dari corpus sekecil ini?
#• Bagaimana solusinya pada penerapan dunia nyata (corpus berskala besar)?
